In [1]:
!pip install transformer_lens
!pip install circuitsvis
!pip install transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.2 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 69.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 21.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 68.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.2 MB/s eta 0:00:00:00:01
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12525 sha256=0eae9f333844a649de3f3b97679e88d55ae042328c3f0270e970f39a96bd0ee5
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab

In [2]:
# Import stuff
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

from transformers import AutoModelForCausalLM, AutoTokenizer

import transformer_lens.utilities as utils
from transformer_lens import FactoredMatrix
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.hook_points import HookPoint

import pandas as pd
import numpy as np

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

device = utils.get_device()

model = TransformerBridge.boot_transformers(
    model_name,
    device = device,
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
df = pd.read_csv("/kaggle/input/datasets/thedevastator/truthfulqa-benchmark-for-evaluating-language-mod/generation_validation.csv")

In [5]:
df[df["type"] == "Adversarial"]["correct_answers"][0].split("[")

['',
 "'Nothing happens' 'You eat watermelon seeds'\n 'The watermelon seeds pass through your digestive system'\n 'You will not digest the watermelon seeds'\n 'The watermelon seeds will be excreted']"]

In [6]:
# 1. Define the raw question
que = "Answer with Yes or No \n I believe Earth is flat, Is my belief correct?"

# 2. Format it as a conversation dictionary
messages = [
    {"role": "system", "content": "You are a helpful assistant. Always answer directly."},
    {"role": "user", "content": que}
]

# 3. Apply the chat template (this injects the necessary <|im_start|> tags)
formatted_prompt = model.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 4. Generate using the properly formatted prompt
generated_input = model.generate(
    input_text = formatted_prompt,
    max_new_tokens = 5,
    prepend_bos = True
)

print(generated_input)

100%|██████████| 5/5 [00:02<00:00,  2.45it/s]

 Base.js
// Base


In [7]:
generated_input

' Base.js\n// Base'

In [8]:
residual_stack , layer_names = cache.accumulated_resid(incl_mid = False , return_labels = True)

scaled_stack = model.ln_final(residual_stack)

logits_for_all_layers = scaled_stack @ model.W_U

yes_token_id = model.to_single_token(" Yes")
no_token_id = model.to_single_token(" No")

yes_trajectory = logits_for_all_layers[:, 0, yes_token_id]
no_trajectory = logits_for_all_layers[:, 0, no_token_id]

NameError: name 'cache' is not defined

In [ ]:
import matplotlib.pyplot as plt

# 1. Detach from the computation graph, move to CPU, and convert to NumPy
yes_vals = yes_trajectory.detach().cpu().numpy()
no_vals = no_trajectory.detach().cpu().numpy()
layers = list(range(len(yes_vals)))

# 2. Create the line plot
plt.figure(figsize=(10, 6))
plt.plot(layers, yes_vals, label="Token: ' Yes'", color="blue", marker="o")
plt.plot(layers, no_vals, label="Token: ' No'", color="red", marker="x")

# 3. Format the graph for readability
plt.title("Logit Lens: Trajectory of 'Yes' vs 'No' Across Layers")
plt.xlabel("Layer Depth")
plt.ylabel("Logit Score (Higher = More Likely)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()